In [1]:
import pickle
with open('./output/store/pdf_files.pkl', 'rb') as f:
    pdf_files, year = pickle.load(f)

In [2]:
print(len(pdf_files))

555


In [3]:
scientific_common_words = {
    "figure", "fig", "table", "et", "al", "etc", "eg", "i.e", "ie", 
    "eq", "conclusion", "introduction", "methods", "study", 
    "research", "author", "authors", "paper", "work", 
    "proposed", "presented", "based", "using", 
    "performed", "obtained", "observed", "used", "shown", "reported", 
    "significant", "important", "novel", "investigation", "algorithm", 
    "algorithms", "technique", "techniques", "parameters", 
    "parameter", "performance", "implemented", "implementation", "similar", 
    "different", "result", "respectively", "compare", "compared", 
    "comparison", "additional", "respectively", " ", "v.", "supra",
    "give", "follow", "know", "high","year", "term", "_",
    "find", "include", "tell" "try", "51ac", "pmpa", "id._note", "item", "coef",
    "interview", "percent", "go", "construct", "scale","000s", "ing", "pricewaterhousecooper",
    "analyse", "dmu?", "qsr?", "consequently", "furthermore", "atmo, id."
}

In [76]:
import re
import spacy

nlp = spacy.load("en_core_web_lg")
nlp.max_length = 3000000

pattern =re.compile(r"^(?=.*[^A-Za-z]).+$")

def filter_doc(doc):
    return [
        token.lemma_.lower() for token in doc
        if (not token.is_stop and 
            not token.is_punct and 
            not token.like_email and 
            not token.like_url and 
            not token.like_num and 
            not token.is_space and
            token.ent_type_ != "PERSON" and 
            token.pos_ != "PROPN" and 
            len(token.text) > 2 and 
            not re.match(pattern, token.text))
    ]

In [5]:
from tqdm import tqdm
doc_objects = list(tqdm(nlp.pipe(pdf_files, n_process=4, batch_size=10), total=len(pdf_files)))

100%|███████████████████████████████████████████████████████████████████████| 555/555 [02:22<00:00,  3.88it/s]


In [77]:
filtered_tokens = [filter_doc(doc) for doc in doc_objects]

all_tokens = []
for tokens in filtered_tokens:
    all_tokens.extend(tokens)
unique_tokens_spacy = set(all_tokens)
print(f"Unique tokens after spaCy filtering: {len(unique_tokens_spacy)}")

Unique tokens after spaCy filtering: 21506


In [78]:
preprocessed_docs = [' '.join(tokens) for tokens in filtered_tokens]
print(preprocessed_docs[0])

equate franchisee employee mail present equate franchisee employee paper present current australian court definition employee independent contractor highlight franchisee fit indistinguishable time employee time independent contractor paper examine policy insolvency legislation query appropriate accord franchisee specific status franchisor insolvency like enjoy employee situation definition relatively unimportant ancillary contract regulate franchisee relationship franchisor insolvent failure law pace franchise business model buy sharp focus employee independent contractor clearly understand right enshrine statute franchisee specific right franchisee creditor franchisor right attend creditor meeting vulnerable franchisee categorize asset liability insolvent estate franchisor franchisee relationship feature franchisee vulnerable employer employer insolvent franchisor insolvent franchisee loose value sink investment right occupy premise able free onerous contract enter position franchisee

In [79]:
from sklearn.feature_extraction.text import CountVectorizer
def analyzer(text):
    return text.split(" ")
vectorizer_model = CountVectorizer(
    analyzer=analyzer,
    min_df=2,
    max_df=0.5,
    ngram_range=(1, 3),
)

In [71]:
vectorizer_model.fit(preprocessed_docs)
vocab = vectorizer_model.get_feature_names_out()
print(f"Unique tokens after CountVectorizer filtering: {len(vocab)}")

Unique tokens after CountVectorizer filtering: 11515


In [72]:
from sentence_transformers import SentenceTransformer, models

model_name = "allenai/scibert_scivocab_uncased"
model = models.Transformer(model_name)
pooling_model = models.Pooling(
    model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)
embedding_model = SentenceTransformer(modules=[model, pooling_model])

In [81]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=10,
    n_components=4,
    min_dist = 0.1,
    metric="cosine",
    n_jobs=-1           
)

In [82]:
import hdbscan

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=10,
    core_dist_n_jobs=-1         
)

In [83]:
from bertopic.representation import MaximalMarginalRelevance
from bertopic import BERTopic
mmr = MaximalMarginalRelevance(diversity=0.5)
topic_model = BERTopic(
    representation_model = mmr, 
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model, 
    verbose=True)
topics, probabilities = topic_model.fit_transform(preprocessed_docs)

2025-03-17 15:14:19,766 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

2025-03-17 15:16:33,770 - BERTopic - Embedding - Completed ✓
2025-03-17 15:16:33,771 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-03-17 15:16:34,223 - BERTopic - Dimensionality - Completed ✓
2025-03-17 15:16:34,224 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-03-17 15:16:34,235 - BERTopic - Cluster - Completed ✓
2025-03-17 15:16:34,236 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-03-17 15:16:37,027 - BERTopic - Representation - Completed ✓


In [84]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,245,-1_analyst_submission_migrant_bottler,"[analyst, submission, migrant, bottler, innova...",[phdassistant rue advantage organizational str...
1,0,57,0_internationalization_disability_inhabitant_s...,"[internationalization, disability, inhabitant,...",[market saturation intangible problem spanish ...
2,1,52,1_covenant_defendant_liable_reasonableness,"[covenant, defendant, liable, reasonableness, ...",[franchise goodwill well email present abstrac...
3,2,52,2_veteran_rhetoric_mentor_proactivity,"[veteran, rhetoric, mentor, proactivity, proac...",[explore theory base abstract use franchise ad...
4,3,35,3_microfoundation_fran_systemwide_appropriability,"[microfoundation, fran, systemwide, appropriab...",[decision make structure mail mail present dec...
5,4,20,4_grievance_disconfirmation_omnichannel_mutuality,"[grievance, disconfirmation, omnichannel, mutu...",[versus versus widely recognize multi unit fra...
6,5,19,5_austrian_conglomerate_drafting_gastronomy,"[austrian, conglomerate, drafting, gastronomy,...",[learn effect case mail mail present learn eff...
7,6,15,6_personalization_triadic_greeting_operant,"[personalization, triadic, greeting, operant, ...",[improve customer satisfaction personalization...
8,7,14,7_reassure_refurbish_filing_freeriding,"[reassure, refurbish, filing, freeriding, deat...",[bank prefer finance franchise business despit...
9,8,13,8_nonprofit_emotion_healthcare_initiator,"[nonprofit, emotion, healthcare, initiator, ki...",[evolution replication chain mail present supp...


In [85]:
topic_info.to_excel("./output/bert_topics.xlsx", index=False)
print("Topic information has been exported to topics.xlsx")

Topic information has been exported to topics.xlsx
